<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
UE L2 Projet de Statistique CMI OSIA
</div>

</a>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Année 2026-2027
</div>
<div style="color: blue; font-size: 1.5em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Nom :  - Prénom :

Nom : - Prénom :
</div>


Modèle de compte-rendu du projet avec codes commentés, sorties numériques et graphiques

**A compléter...**

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des données
</div>

Le code ci-dessous permet d'importer ces différents fichiers, de visualiser leur contenu ainsi que leur organisation sous forme de DataFrame Pandas.

**Noms et coordonnées des stations**

In [ ]:
import pandas as pd

stations_toulouse = pd.read_parquet("stations_toulouse.parquet")

**Données d'entrainement**

``history_toulouse_train`` dans le code ci-dessous est un DataFrame Pandas, avec :
- des lignes identifiées par un MultiIndex (nom de la station et date de mesure)
- des colonnes nommées ``bikes`` et ``stands``qui contiennent respectivement le nombre de vélos disponibles et le nombre de vélos manquants pour une station et une date (jour/heure) données

In [ ]:
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")

**Données de test**

``history_toulouse_test`` dans le code ci-dessous est un DataFrame Pandas, avec :
- des lignes identifiées par un MultiIndex (une station à la date de la dernière mesure faite dans les données d'entraînement)
- des colonnes nommées ``bikes_+15min``,...,``bikes_+120min`` qui doivent être complétées par des prévisions tous les 15 minutes pendant 2 heures

In [ ]:
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Statistique descriptive des données
</div>

Vous devez inclure ici les résultats de statistique descriptive qui vous semblent pertinents pour le problème de prévision du nombre de vélos. Vous devez en particulier répondre aux questions suivantes :

- (a) déterminer le nombre maximal de vélos disponibles par station et afficher sur une carte les stations en fonction d'un code couleur proportionnel à ce nombre
- (b) déterminer s'il existe des valeurs manquantes dans les données
- (c) déterminer les nombres de mesures pour chaque station et afficher un histogramme de leur répartition
- (d) déterminer les dates de début et de fin des mesures par stations, et faire une représentation graphique de leur dispersion
- (e) proposer des statistique de base par station : maximimum, minimum, moyenne, écart-type, quartiles, et en faire une représentation par Boxplot
- (f) calculer la courbe moyenne journalière de remplissage d'une station (toutes les 15 minutes), et de l'ensemble des stations / calculer également la courbe médiane, 1er-quartile, 3-ème quartile
- (g) tracer sur quelques cartes l'évolution temporelle de remplissage moyen de l'ensemble des stations en choisissant quelques moments dans une journée et en déduire des dynamiques urbaines de remplissage en fonction des heures de la journée et de la localisation des stations
- (h) autres idées ?...

**A compléter...**

<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Méthodes de prévision basiques
</div>

Vous devez inclure ici une description des méthodes de bases que vous avez explorées, ainsi qu'une évaluation de leurs performances (cacul de l'erreur RMSE) sur un jeu de données de validation créé à partir de l'ensemble d'apprentissage.

Il est attendu :

- une proposition de 3 méthodes de prévision
- leur classement en terme de performances par calcul de l'erreur RMSE sur l'ensemble de validation
- leurs prévisions sur l'ensemble test à déposer sur Moodle

**Création d'un ensemble de validation**

Le principe de création d'un ensemble de validation est le suivant. Les données disponibles sont partagées en deux sous-ensembles disjoints :

- un ensemble d'apprentissage qui va servir à créer un modèle de prévision
- un ensemble de validation qui va servir à évaluer le modèle de prévision

Il s'agit d'un principe fondamental en apprentissage statistique qui consiste à construire une méthode d'estimation puis **la tester sur un jeu de données qui n'a pas servi à sa construction**.

Dans ce projet, pour créer l'ensemble de validation, on peut simplement retirer les 8 denières mesures dans chaque station de ``history_toulouse_train`` et conserver le reste des données pour l'apprentissage.

In [ ]:
# Exemple de création d'un jeu de données d'apprentissage et un jeu de données de validation

# Création ensemble d'apprentissage
# Suppression des 8 dernières mesures de chaque station
rang_depuis_la_fin = (
    history_toulouse_train
    .groupby(level="station")
    .cumcount(ascending=False)
)
rang_depuis_la_fin

In [ ]:
history_toulouse_apprentissage = history_toulouse_train[
    rang_depuis_la_fin >= 8
].copy()
history_toulouse_apprentissage

In [ ]:
# Récupèration de la dernière date de mesure pour chaque station
# des données d'apprentissage
last_apprentissage = (
    history_toulouse_apprentissage
    .groupby(level="station")
    .tail(1)
    .copy()
)

last_apprentissage

In [ ]:
# Creation de l'ensemble de validation
# On récupère les 9 dernières mesures :
# - la 1ère = dernière mesure dans l'ensemble d'apprentissage
# - les 8 suivantes = valeurs à prédire
last_9 = (
    history_toulouse_train
    .groupby(level="station")
    .tail(9)
    .copy()
)
last_9

In [ ]:
# On remet l'index sous forme de colonnes
last_9 = last_9.reset_index()
last_9

In [ ]:
# Numéro de position parmi les 9 dernières mesures
last_9
    

In [ ]:
last_9["horizon"] = (
    last_9.groupby("station").cumcount()
)
last_9

In [ ]:
# Recherche de valeurs manquantes
missing = last_9.loc[
    last_9[['bikes', 'stands']].isna().any(axis=1),
    ['station', 'commit_at', 'bikes', 'stands']
]

print(missing)

In [ ]:
# On crée une ligne par station et on récupère les 8 valeurs futures
history_toulouse_validation = (
    last_9
    .pivot(
        index="station",
        columns="horizon",
        values="bikes"
    )
    .drop(columns=0)
    .rename(columns={
        1: "bikes_+15min",
        2: "bikes_+30min",
        3: "bikes_+45min",
        4: "bikes_+60min",
        5: "bikes_+75min",
        6: "bikes_+90min",
        7: "bikes_+105min",
        8: "bikes_+120min"
    })
)

# Récupérer la date de la dernière mesure de l'ensemble d'apprentissage par station
cutoff_times = (
    last_9[last_9["horizon"] == 0]
    .set_index("station")["commit_at"]
)

# Ajouter commit_at
history_toulouse_validation["commit_at"] = cutoff_times

# Mettre station + commit_at en MultiIndex
history_toulouse_validation = (
    history_toulouse_validation
    .reset_index()
    .set_index(["station", "commit_at"])
    .sort_index()
)

history_toulouse_validation

**Exemple d'évaluation d'une méthode de prévision**

On donne ci-dessous l'exemple du calcul de l'erreur RMSE sur l'ensemble de validation d'une méthode de base à partir de la moyenne des vélos par station sur l'ensemble d'apprentissage

In [ ]:
# Creation des données à prédire
history_toulouse_basique = history_toulouse_validation.copy()

# Methode de base
mean_bikes_station = (
    history_toulouse_apprentissage
    .groupby(level="station")["bikes"]
    .mean()
    .round()
)

target_columns = [
    "bikes_+15min",
    "bikes_+30min",
    "bikes_+45min",
    "bikes_+60min",
    "bikes_+75min",
    "bikes_+90min",
    "bikes_+105min",
    "bikes_+120min"
]

# Station correspondant à chaque ligne de l'ensemble de validation
stations_a_predire = history_toulouse_basique.index.get_level_values("station")

# Moyenne de chaque station à prédire
predictions = mean_bikes_station.reindex(stations_a_predire)

# Remplissage des 8 horizons avec la même moyenne
history_toulouse_basique.loc[:, target_columns] = (
    predictions.to_numpy()[:, None]
)

In [ ]:
history_toulouse_basique

In [ ]:
history_toulouse_validation

In [ ]:
# Recherche de valeurs manquantes
print("Nombre de valeurs manquantes :", predictions.isna().sum())
print("Valeurs manquantes :", predictions[predictions.isna()])

In [ ]:
# Gestion des valeurs manquantes
stations_problematiques = stations_a_predire[predictions.isna()]
print(stations_problematiques)

In [ ]:
# Nombre d'observations par station
nb_obs = (
    history_toulouse_train
    .groupby(level="station")
    .count()
)
nb_obs

In [ ]:
import numpy as np
np.min(nb_obs)

In [ ]:
print(nb_obs.loc[stations_problematiques])

In [ ]:
vect_nb_obs = nb_obs["bikes"]

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(vect_nb_obs, bins=20)

plt.xlabel("Nombre d'observations par station")
plt.ylabel("Nombre de stations")
plt.title("Distribution du nombre d'observations par station")

plt.show()

In [ ]:
# Stations sans valeurs manquantes
stations_sans_NaN = stations_a_predire[predictions.notna()]
print(stations_sans_NaN)

In [ ]:
import numpy as np

history_toulouse_validation.values

In [ ]:
# Calcul de l'erreur RMSE sur l'ensemble de validation
rmse = np.sqrt((
    (history_toulouse_basique.values- history_toulouse_validation.values)**2
).mean())
print(f'RMSE',rmse)

In [ ]:
# Calcul de l'erreur RMSE sur l'ensemble de validation
rmse = np.sqrt((
    (history_toulouse_basique.loc[stations_sans_NaN].values- history_toulouse_validation.loc[stations_sans_NaN].values)**2
).mean())
print(f'RMSE',rmse)

<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Prévision par apprentissage statistique
</div>

A venir...